In [ ]:
from app.utils.langfuse.client import langfuse
from app.utils.langfuse.integration import span
from app.tools.llm_client import GigaChatConnector
from langchain_openai import ChatOpenAI

from langfuse import Evaluation

import pandas as pd
import json
import re
import os

import logging
from src.common.config.config import settings
logging.disable(logging.CRITICAL)

In [ ]:
# llm = GigaChatConnector(
#     model_name='GigaChat-2-Max',
#     temperature=1,
#     top_p=0,
# )

llm = ChatOpenAI(
    model='gpt-oss-120b',
    api_key=settings.get("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
    temperature=0,
)

In [ ]:
dataset = langfuse.get_dataset('legislation_monitoring_test_data_cleaned')
dataset_items = dataset.items

## Evaluators

In [ ]:
evaluation_prompt = """Сравни два списка {field_type}.

Эталон (gold): {gold_items}
Предсказание (pred): {pred_items}

Для каждого элемента из эталона определи, есть ли семантически эквивалентный элемент в предсказании.
Учитывай синонимы, сокращения и разные формулировки одного понятия.
Например: "КоАП" и "кодекс об административных правонарушениях" - это одно и то же.

Ответь в формате JSON:
{{
    "gold_found": ["элемент1", ...],
    "gold_missed": ["элемент2", ...],
    "pred_extra": ["элемент3", ...]
}}"""

In [ ]:
def normalize_to_list(value) -> list[str]:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return []
    if isinstance(value, list):
        return [str(v).lower().strip() for v in value if v]
    return [s.strip().lower() for s in str(value).split(',') if s.strip()]


def exact_match_bool(pred, gold) -> bool:
    if gold is None or (isinstance(gold, float) and pd.isna(gold)):
        gold_bool = False
    else:
        gold_bool = bool(gold)
    pred_bool = bool(pred)
    return pred_bool == gold_bool


def fuzzy_contains(needle: str, haystack: str) -> bool:
    needle_words = set(needle.split())
    haystack_words = set(haystack.split())
    if len(needle_words) <= 2:
        return needle in haystack or haystack in needle
    common = needle_words & haystack_words
    return len(common) >= len(needle_words) * 0.6


def find_exact_matches(pred_items: list[str], gold_items: list[str]) -> tuple[list[str], list[str], list[str]]:
    gold_found = []
    gold_remaining = list(gold_items)
    pred_remaining = list(pred_items)

    for gold in list(gold_remaining):
        for pred in list(pred_remaining):
            if gold == pred or fuzzy_contains(gold, pred) or fuzzy_contains(pred, gold):
                gold_found.append(gold)
                gold_remaining.remove(gold)
                pred_remaining.remove(pred)
                break

    return gold_found, gold_remaining, pred_remaining


def llm_semantic_match(pred_items: list[str], gold_items: list[str], field_type: str) -> dict:
    if not gold_items:
        return {
            'recall': 1.0,
            'precision': 1.0 if not pred_items else 0.0,
            'gold_found': [],
            'gold_missed': [],
            'pred_extra': pred_items,
        }

    if not pred_items:
        return {
            'recall': 0.0,
            'precision': 0.0,
            'gold_found': [],
            'gold_missed': gold_items,
            'pred_extra': [],
        }

    exact_found, gold_remaining, pred_remaining = find_exact_matches(pred_items, gold_items)

    if not gold_remaining:
        return {
            'recall': 1.0,
            'precision': len(exact_found) / len(pred_items) if pred_items else 1.0,
            'gold_found': exact_found,
            'gold_missed': [],
            'pred_extra': pred_remaining,
        }

    if not pred_remaining:
        return {
            'recall': len(exact_found) / len(gold_items),
            'precision': 1.0,
            'gold_found': exact_found,
            'gold_missed': gold_remaining,
            'pred_extra': [],
        }

    prompt = evaluation_prompt.format(
        field_type=field_type,
        gold_items=gold_remaining,
        pred_items=pred_remaining,
    )

    messages = [("user", prompt)]
    response = llm.invoke(messages)

    try:
        json_match = re.search(r'\{[^{}]*\}', response, re.DOTALL)
        if json_match:
            result = json.loads(json_match.group())
        else:
            result = json.loads(response)
    except Exception:
        result = {
            'gold_found': [],
            'gold_missed': gold_remaining,
            'pred_extra': pred_remaining,
        }

    llm_gold_found = result.get('gold_found', [])
    llm_gold_missed = result.get('gold_missed', [])
    llm_pred_extra = result.get('pred_extra', [])

    all_gold_found = exact_found + llm_gold_found

    recall = len(all_gold_found) / len(gold_items) if gold_items else 1.0
    total_matched = len(all_gold_found)
    precision = total_matched / len(pred_items) if pred_items else 1.0

    return {
        'recall': recall,
        'precision': precision,
        'gold_found': all_gold_found,
        'gold_missed': llm_gold_missed,
        'pred_extra': llm_pred_extra,
    }



In [ ]:
def has_keywords_evaluator(*, input, output, expected_output, metadata=None, **kwargs):
    pred = output or {}
    gold = expected_output or {}
    match = exact_match_bool(pred.get('has_keywords'), gold.get('has_keywords'))
    value = 1.0 if match else 0.0
    comment = f"pred={pred.get('has_keywords')}, gold={gold.get('has_keywords')}"
    return Evaluation(name="has_keywords_accuracy", value=value, comment=comment)


def has_laws_evaluator(*, input, output, expected_output, metadata=None, **kwargs):
    pred = output or {}
    gold = expected_output or {}
    match = exact_match_bool(pred.get('has_laws'), gold.get('has_laws'))
    value = 1.0 if match else 0.0
    comment = f"pred={pred.get('has_laws')}, gold={gold.get('has_laws')}"
    return Evaluation(name="has_laws_accuracy", value=value, comment=comment)


def priority_evaluator(*, input, output, expected_output, metadata=None, **kwargs):
    pred = output or {}
    gold = expected_output or {}
    pred_priority = pred.get('priority')
    gold_priority = gold.get('priority')
    if gold_priority is not None and not (isinstance(gold_priority, float) and pd.isna(gold_priority)):
        try:
            gold_int = int(gold_priority)
        except Exception:
            gold_int = gold_priority
        match = pred_priority == gold_int
    else:
        match = pred_priority is None
    value = 1.0 if match else 0.0
    comment = f"pred={pred_priority}, gold={gold_priority}"
    return Evaluation(name="priority_accuracy", value=value, comment=comment)

In [ ]:
def keywords_recall_evaluator(*, input, output, expected_output, metadata=None, **kwargs):
    pred = output or {}
    gold = expected_output or {}
    pred_keywords = normalize_to_list(pred.get('keywords'))
    gold_keywords = normalize_to_list(gold.get('keywords'))
    eval_res = llm_semantic_match(pred_keywords, gold_keywords, 'ключевых слов')
    value = float(eval_res['recall'])
    comment = f"found={eval_res['gold_found']}, missed={eval_res['gold_missed']}, extra={eval_res['pred_extra']}"
    return Evaluation(name="keywords_recall", value=value, comment=comment)


def keywords_precision_evaluator(*, input, output, expected_output, metadata=None, **kwargs):
    pred = output or {}
    gold = expected_output or {}
    pred_keywords = normalize_to_list(pred.get('keywords'))
    gold_keywords = normalize_to_list(gold.get('keywords'))
    eval_res = llm_semantic_match(pred_keywords, gold_keywords, 'ключевых слов')
    value = float(eval_res['precision'])
    comment = f"found={eval_res['gold_found']}, missed={eval_res['gold_missed']}, extra={eval_res['pred_extra']}"
    return Evaluation(name="keywords_precision", value=value, comment=comment)


def laws_recall_evaluator(*, input, output, expected_output, metadata=None, **kwargs):
    pred = output or {}
    gold = expected_output or {}
    pred_laws = normalize_to_list(pred.get('laws'))
    gold_laws = normalize_to_list(gold.get('laws'))
    eval_res = llm_semantic_match(pred_laws, gold_laws, 'законов')
    value = float(eval_res['recall'])
    comment = f"found={eval_res['gold_found']}, missed={eval_res['gold_missed']}, extra={eval_res['pred_extra']}"
    return Evaluation(name="laws_recall", value=value, comment=comment)


def laws_precision_evaluator(*, input, output, expected_output, metadata=None, **kwargs):
    pred = output or {}
    gold = expected_output or {}
    pred_laws = normalize_to_list(pred.get('laws'))
    gold_laws = normalize_to_list(gold.get('laws'))
    eval_res = llm_semantic_match(pred_laws, gold_laws, 'законов')
    value = float(eval_res['precision'])
    comment = f"found={eval_res['gold_found']}, missed={eval_res['gold_missed']}, extra={eval_res['pred_extra']}"
    return Evaluation(name="laws_precision", value=value, comment=comment)


def subject_recall_evaluator(*, input, output, expected_output, metadata=None, **kwargs):
    pred = output or {}
    gold = expected_output or {}
    pred_subject = normalize_to_list(pred.get('subject'))
    gold_subject = normalize_to_list(gold.get('subject'))
    eval_res = llm_semantic_match(pred_subject, gold_subject, 'субъектов законодательной инициативы')
    value = float(eval_res['recall'])
    comment = f"found={eval_res['gold_found']}, missed={eval_res['gold_missed']}, extra={eval_res['pred_extra']}"
    return Evaluation(name="subject_recall", value=value, comment=comment)


def subject_precision_evaluator(*, input, output, expected_output, metadata=None, **kwargs):
    pred = output or {}
    gold = expected_output or {}
    pred_subject = normalize_to_list(pred.get('subject'))
    gold_subject = normalize_to_list(gold.get('subject'))
    eval_res = llm_semantic_match(pred_subject, gold_subject, 'субъектов законодательной инициативы')
    value = float(eval_res['precision'])
    comment = f"found={eval_res['gold_found']}, missed={eval_res['gold_missed']}, extra={eval_res['pred_extra']}"
    return Evaluation(name="subject_precision", value=value, comment=comment)

In [ ]:
evaluators = [
    has_keywords_evaluator,
    has_laws_evaluator,
    priority_evaluator,
    keywords_recall_evaluator,
    keywords_precision_evaluator,
    laws_recall_evaluator,
    laws_precision_evaluator,
    subject_recall_evaluator,
    subject_precision_evaluator,
]

## Run-level eval

### Полнота

In [ ]:
def laws_recall_general_evaluator(*, item_results, **kwargs):
    gen_laws_recall = []
    for result in item_results:
        for eval in result.evaluations:
            if eval.name == "laws_recall":
                gen_laws_recall.append(eval.value)

    if not gen_laws_recall:
        return Evaluation(name="laws_recall", value=None)
 
    avg = sum(gen_laws_recall) / len(gen_laws_recall)
 
    return Evaluation(
        name="Полнота по законам", 
        value=avg, 
        # comment=f"Average accuracy: {avg:.2%}"
    )

def keywords_recall_general_evaluator(*, item_results, **kwargs):
    gen_keywords_recall = []
    for result in item_results:
        for eval in result.evaluations:
            if eval.name == "keywords_recall":
                gen_keywords_recall.append(eval.value)

    if not gen_keywords_recall:
        return Evaluation(name="keywords_recall", value=None)
 
    avg = sum(gen_keywords_recall) / len(gen_keywords_recall)
 
    return Evaluation(
        name="Полнота по ключевым словам", 
        value=avg, 
        # comment=f"Average accuracy: {avg:.2%}"
    )


def subject_recall_general_evaluator(*, item_results, **kwargs):
    gen_subject_recall = []
    for result in item_results:
        for eval in result.evaluations:
            if eval.name == "subject_recall":
                gen_subject_recall.append(eval.value)

    if not gen_subject_recall:
        return Evaluation(name="subject_recall", value=None)
 
    avg = sum(gen_subject_recall) / len(gen_subject_recall)
 
    return Evaluation(
        name="Полнота по субъектам", 
        value=avg, 
        # comment=f"Average accuracy: {avg:.2%}"
    )

In [ ]:
def priority_accuracy_general_evaluator(*, item_results, **kwargs):
    accuracies = [
        eval.value for result in item_results
        for eval in result.evaluations
        if eval.name == "priority_accuracy"
    ]
 
    if not accuracies:
        return Evaluation(name="priority_accuracy", value=None)
 
    avg = sum(accuracies) / len(accuracies)
 
    return Evaluation(
        name="Точность приоритета", 
        value=avg, 
        # comment=f"Average accuracy: {avg:.2%}"
    )

def has_keywords_accuracy_general_evaluator(*, item_results, **kwargs):
    accuracies = [
        eval.value for result in item_results
        for eval in result.evaluations
        if eval.name == "has_keywords_accuracy"
    ]
 
    if not accuracies:
        return Evaluation(name="has_keywords_accuracy", value=None)
 
    avg = sum(accuracies) / len(accuracies)
 
    return Evaluation(
        name="Точность по ключевым словам", 
        value=avg, 
        # comment=f"Average accuracy: {avg:.2%}"
    )

def has_laws_accuracy_general_evaluator(*, item_results, **kwargs):
    accuracies = [
        eval.value for result in item_results
        for eval in result.evaluations
        if eval.name == "has_laws_accuracy"
    ]
 
    if not accuracies:
        return Evaluation(name="has_laws_accuracy", value=None)
 
    avg = sum(accuracies) / len(accuracies)
 
    return Evaluation(
        name="Точность по законам", 
        value=avg, 
        # comment=f"Average accuracy: {avg:.2%}"
    )

## Подготовка пайплайна для оценки

In [ ]:
from app.main import analyze_bill

In [ ]:
def get_formatted_results(url):
    try:
        result = analyze_bill(url)
        keywords = list(set([k['keyword'].lower() for k in result['keywords_result']['found_keywords']]))
        laws = list(set([l.lower() for l in result['law_changes_result']['affected_laws']]))
        final_result = {
            'url': result['card_url'],
            'keywords': keywords,
            'has_keywords': result['keywords_result']['keywords_found'],
            'laws': laws,
            'has_laws': result['law_changes_result']['affects_target_laws'],
            'priority': result['priority_result']['priority'],
            'subject': result['initiators'] if len(result['initiators']) > 1 else result['initiators'][0],
            'summary': result['summary_result']['summary_text'],
        }
        return final_result
    except Exception as e:
        print(f'Error: {e}')
        final_result = {
            'url': url,
            'keywords': [],
            'has_keywords': False,
            'laws': [],
            'has_laws': False,
            'priority': None,
            'subject': None,
            'summary': f'Ошибка - {e}',
        }
        return final_result

## Eval

In [ ]:
results = {}

In [ ]:
from tqdm import tqdm

for item in tqdm(dataset_items, desc="Обработка законопроектов"):
    url = item.input['url']
    results[url] = get_formatted_results(url)

In [ ]:
def my_task(*, item, **kwargs):
    url = item.input['url']
    result = results[url]
    return result

In [ ]:
experiment_results = dataset.run_experiment(
    name="Мониторинг законодательства - оценка качества на первом тестовом датасете от заказчика",
    # description="Monthly evaluation of our production model",
    task=my_task, # see above for the task definition
    evaluators=[
        has_keywords_evaluator,
        has_laws_evaluator,
        priority_evaluator,
        keywords_recall_evaluator,
        keywords_precision_evaluator,
        laws_recall_evaluator,
        laws_precision_evaluator,
        subject_recall_evaluator,
        subject_precision_evaluator,
    ],
    run_evaluators=[
        laws_recall_general_evaluator,
        keywords_recall_general_evaluator,
        subject_recall_general_evaluator,
        priority_accuracy_general_evaluator,
        has_keywords_accuracy_general_evaluator,
        has_laws_accuracy_general_evaluator
    ]
)

In [ ]:
from pprint import pprint
pprint(experiment_results.format())